# Лабораторная работа: Машина опорных векторов (SVM)

<img src="svm_img.png" width=350>

Машина опорных векторов (Support Vector Machine, SVM) — один из классических алгоритмов машинного обучения. В этой лабораторной вы **построите SVM с нуля**: сведёте обучение модели к задаче квадратичного программирования, решите её готовым солвером, реализуете ядровой SVM и, наконец, примените свою реализацию к реальным медицинским данным (диагностика болезней сердца).

**Маршрут работы:**

1. **Линейное и квадратичное программирование** — разберём задачи линейного (LP) и квадратичного (QP) программирования, поймём, чем они отличаются, и научимся решать их библиотекой `cvxopt`.
2. **Линейный SVM** — разберём теорию и реализуем линейный SVM через QP-солвер.
3. **Двойственный переход и ядровой SVM** — перепишем задачу в двойственной форме и научимся строить нелинейные разделяющие поверхности с помощью ядер.
4. **Реальные данные** — возьмём датасет UCI о болезнях сердца, подготовим его средствами `pandas` и запустим на нём собственную реализацию SVM.

Результат лабораторной работы — **отчёт** в формате ноутбука IPython (ipynb-файл). Нам не интересен ваш код — важны рассуждения и эксперименты. Чем меньше кода, тем лучше всем: нам — меньше проверять, вам — проще найти ошибку или дополнить эксперимент.

Постарайтесь сделать ваш отчёт интересным рассказом, последовательно отвечающим на вопросы из заданий. Ответы на вопросы должны быть полными, чёткими и хорошо аргументированными.

## 0. Подготовка ноутбука

Подключим библиотеки, которые понадобятся на протяжении всей работы:

- [`NumPy`](https://numpy.org/) — операции с массивами и матрицами;
- [`Matplotlib`](https://matplotlib.org/) — графики;
- [`cvxopt`](https://cvxopt.org/) — солвер задач квадратичного программирования;
- [`scikit-learn`](https://scikit-learn.org/) — генерация выборок, разбиение данных, эталонные модели для сравнения;
- [`pandas`](https://pandas.pydata.org/) — работа с реальными табличными данными (часть 4).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['figure.figsize'] = (6, 4)
plt.rcParams['font.size'] = 12

# фиксируем seed для воспроизводимости случайных экспериментов
np.random.seed(42)

In [ ]:
# cvxopt не входит в стандартный набор Anaconda.
# Если библиотека не установлена — раскомментируйте следующую строку и выполните ячейку:
# !pip install cvxopt

In [ ]:
from cvxopt import matrix, solvers

## 1. Линейное и квадратичное программирование

### 1.1. Линейное программирование: плоская цель

Начнём с самого простого типа задач оптимизации с ограничениями. В **линейном программировании** (LP) и целевая функция, и все ограничения линейны:

$$
\boldsymbol c^T \boldsymbol x \to \min_{\boldsymbol x}
\qquad
\text{при условиях: } \
\boldsymbol G\boldsymbol x \le \boldsymbol h, \qquad \boldsymbol A\boldsymbol x = \boldsymbol b
$$

Интуиция: целевая функция — не «чаша», а **наклонная плоскость**. Допустимая область, которую задают линейные ограничения, — выпуклый многоугольник (в многомерном случае — многогранник). Ключевой факт, который вы увидите на картинке ниже: **минимум линейной функции всегда достигается на границе — в вершине многоугольника** (либо вдоль целого ребра, если оно параллельно линиям уровня). «Внутрь» области оптимум не попадает никогда — плоской функции там двигаться некуда.

Откуда берутся такие задачи: LP — математика планирования: оптимальный план производства, рацион минимальной стоимости, транспортная задача. Слово «программирование» здесь означает «планирование», а не написание кода.

В cvxopt есть солвер и для LP: [`solvers.lp(c, G, h)`](https://cvxopt.org/examples/tutorial/linearprogramming.html) — интерфейс почти тот же, что у QP-солвера, только вместо матрицы $P$ передаётся вектор $\boldsymbol c$.

### 1.2. Пример: план кондитерской

Кондитерская печёт песочные ($x_1$ шт.) и бисквитные ($x_2$ шт.) пирожные. Прибыль — 30 и 40 рублей за штуку. Печь работает с загрузкой: песочное занимает 1 час, бисквитное — 2 часа, всего доступно 8 часов. Время кондитера: 3 часа на партию песочных, 2 часа — на бисквитные, всего 12 часов. Сколько печь, чтобы прибыль была максимальной?

$$
30x_1 + 40x_2 \to \max
\qquad
\text{при: } \
x_1 + 2x_2 \le 8, \quad 3x_1 + 2x_2 \le 12, \quad x_1 \ge 0, \ x_2 \ge 0
$$

Солвер умеет только **минимизировать**, поэтому максимизацию заменим минимизацией цели со знаком минус: $\boldsymbol c = (-30, -40)$:

In [ ]:
c = matrix(np.array([-30.0, -40.0]), tc='d')

G = matrix(np.array([[ 1.0,  2.0],    #  x1 + 2x2 <= 8    (печь)
                     [ 3.0,  2.0],    #  3x1 + 2x2 <= 12  (кондитер)
                     [-1.0,  0.0],    # -x1        <= 0
                     [ 0.0, -1.0]]), tc='d')   # -x2 <= 0
h = matrix(np.array([8.0, 12.0, 0.0, 0.0]), tc='d')

sol_lp = solvers.lp(c, G, h)
print('x* =', np.array(sol_lp['x']).ravel())
print('прибыль =', -sol_lp['primal objective'])

# для самопроверки: должно получиться x* = (2, 3) и прибыль 180

### 1.3. Визуализация: оптимум — в вершине

Нарисуем допустимую область (серым), линии уровня прибыли $30x_1 + 40x_2$ (синий пунктир) и отметим вершины многоугольника. Оптимум — та вершина, через которую линии уровня «проходят последними» при росте прибыли. Приём с булевыми массивами — тот же, что и в визуализации QP ниже:

In [ ]:
x1 = np.linspace(-0.5, 5.0, 200)
x2 = np.linspace(-0.5, 5.0, 200)
xx1, xx2 = np.meshgrid(x1, x2)

profit = 30*xx1 + 40*xx2
ok = (xx1 + 2*xx2 <= 8) & (3*xx1 + 2*xx2 <= 12) & (xx1 >= 0) & (xx2 >= 0)

vertices = np.array([[0, 0], [4, 0], [2, 3], [0, 4]], dtype=float)

plt.figure(figsize=(5.5, 5.5))
plt.grid(True)
plt.xlabel('$x_1$')
plt.ylabel('$x_2$')
plt.contourf(x1, x2, ok, 1, alpha=0.4, cmap='gray')
plt.contour(x1, x2, profit, levels=np.arange(0, 240, 30), colors='tab:blue',
            linestyles='--', linewidths=1)
plt.scatter(vertices[:, 0], vertices[:, 1], color='black', s=40, zorder=3)
plt.scatter([2], [3], color='red', s=160, marker='*', zorder=4)  # оптимум
plt.annotate('оптимум (2, 3)', (2, 3), xytext=(2.7, 3.5),
             arrowprops=dict(arrowstyle='->'))
plt.show()

### 1.4. Квадратичное программирование: чаша вместо плоскости

Теперь усложним цель. В **квадратичном программировании** (QP) минимизируется квадратичная функция — многомерная парабола — при тех же линейных ограничениях:

$$
\frac{1}{2}\boldsymbol x^T\boldsymbol P\boldsymbol x + \boldsymbol q^T\boldsymbol x \to \min_{\boldsymbol x}
\qquad
\text{при условиях: } \
\boldsymbol G\boldsymbol x \le \boldsymbol h, \qquad \boldsymbol A\boldsymbol x = \boldsymbol b
$$

Представьте чашу — график квадратичной функции: теперь у функции появилась собственная «низина», и ограничения лишь мешают в неё попасть. **Минимум может оказаться и внутри области** (если низина допустима), и на её границе — какой из случаев случится, заранее неизвестно. Если матрица $P$ положительно определена ($P \succ 0$), чаша строго выпуклая, и решение единственно.

Разберём обозначения:

| Объект | Размер | Смысл |
|---|---|---|
| $\boldsymbol x$ | $n$ | вектор переменных, которые мы подбираем |
| $\boldsymbol P$ | $n \times n$ | симметричная матрица, задающая «кривизну» чаши |
| $\boldsymbol q$ | $n$ | линейная часть, «наклоняющая» чашу |
| $\boldsymbol G\boldsymbol x \le \boldsymbol h$ | $m \times n$ | неравенства — «заборы»; каждое неравенство — отдельная строка |
| $\boldsymbol A\boldsymbol x = \boldsymbol b$ | $p \times n$ | равенства (в наших задачах их не будет) |

Сводим различия LP и QP в таблицу:

| | LP | QP |
|---|---|---|
| Целевая функция | линейная $\boldsymbol c^T\boldsymbol x$ — «наклонная плоскость» | квадратичная $\frac{1}{2}\boldsymbol x^T P\boldsymbol x + \boldsymbol q^T\boldsymbol x$ — «чаша» |
| Где может быть оптимум | только на границе: вершина или ребро | где угодно: внутри области или на границе |
| Единственность | не гарантирована (оптимум — целое ребро) | при $P \succ 0$ решение единственно |
| Солвер cvxopt | `solvers.lp(c, G, h)` | `solvers.qp(P, q, G, h)` |
| Типичные задачи | планы производства, рацион, транспорт | SVM, инвестиционный портфель, регрессия с ограничениями |

Почему SVM — это именно QP, станет ясно в разделе 2: ширина разделяющего коридора пропорциональна $1/\lVert\boldsymbol w\rVert$, и максимизировать её удобно квадратичным штрафом $\lVert\boldsymbol w\rVert^2$ при линейных ограничениях — линейной цели такой трик не под силу.

Задача QP хорошо изучена; помимо [CVXOPT](https://cvxopt.org/) (функция [`solvers.qp()`](https://cvxopt.org/examples/tutorial/qp.html)) существуют [Mosek](https://www.mosek.com/) (коммерческая, есть бесплатная академическая лицензия) и Matlab Optimization Toolbox (`quadprog`) — использовать можно любой.

### 1.5. Знакомство с `cvxopt`

Солверы `lp` и `qp` принимают ограничения в одинаковом виде, так что всё, что вы узнаете про матрицы ниже, работает в обоих случаях. Главный подводный камень: `cvxopt.matrix` заполняет данные **по столбцам**, а не по строкам, как привычный `NumPy`. Поэтому надёжнее всего создавать матрицы из готовых `numpy`-массивов — тогда сюрпризов не будет:

In [ ]:
# Внимание: список из шести чисел превратился в матрицу 2x3,
# заполненную ПО СТОЛБЦАМ: первый столбец (1, 2), второй (3, 4), третий (5, 6)
A = matrix([1.0, 2.0, 3.0, 4.0, 5.0, 6.0], (2, 3))
print(A)
print('размер матрицы:', A.size)

# Проще передавать numpy-массивы с параметром tc='d' (числа двойной точности)
B = matrix(np.array([[1.0, 2.0, 3.0],
                     [4.0, 5.0, 6.0]]), tc='d')
print(B)

### 1.6. Первый запуск QP-солвера

Решим учебную задачу из [документации cvxopt](https://cvxopt.org/examples/tutorial/qp.html):

$$
\frac{1}{2}x_1^2 + 3x_1 + 4x_2 \to \min
\qquad
\text{при: } \
x_1 \ge 0, \ x_2 \ge 0, \ x_1 + 3x_2 \ge 15, \ 2x_1 + 5x_2 \le 100, \ 3x_1 + 4x_2 \le 80
$$

Квадратичная часть здесь задана матрицей $P = \begin{pmatrix} 1 & 0 \\ 0 & 0 \end{pmatrix}$, линейная — вектором $q = (3, 4)^T$. Дальше каждое неравенство нужно переписать в каноническом виде $g^T x \le h$, например: $x_1 \ge 0 \Leftrightarrow -x_1 \le 0$, а $x_1 + 3x_2 \ge 15 \Leftrightarrow -x_1 - 3x_2 \le -15$. Из получившихся строк-неравенств складываем матрицы $G$ и $h$:

In [ ]:
P = matrix(np.diag([1.0, 0.0]), tc='d')
q = matrix(np.array([3.0, 4.0]), tc='d')

G = matrix(np.array([[-1.0,  0.0],    # -x1         <= 0     (это x1 >= 0)
                     [ 0.0, -1.0],    # -x2         <= 0     (это x2 >= 0)
                     [-1.0, -3.0],    # -x1 - 3x2   <= -15   (это x1 + 3x2 >= 15)
                     [ 2.0,  5.0],    #  2x1 + 5x2  <= 100
                     [ 3.0,  4.0]]), tc='d')   #  3x1 + 4x2 <= 80
h = matrix(np.array([0.0, 0.0, -15.0, 100.0, 80.0]), tc='d')

sol = solvers.qp(P, q, G, h)
print('x* =', np.array(sol['x']).ravel())
print('f(x*) =', sol['primal objective'])
# для самопроверки: должно получиться примерно x* = (0, 5) и f(x*) = 20

# чтобы солвер не печатал подробный лог при каждом вызове, выполните:
# solvers.options['show_progress'] = False

### 1.7. Активные и неактивные ограничения

Ограничение-неравенство называется **активным** в точке оптимума $\boldsymbol x_\star$, если оно выполнено как точное равенство: $g^T \boldsymbol x_\star = h$. Образно: это тот «забор», которого минимум коснулся. Ограничение, до которого минимум «не дотянулся», — неактивное: его можно убрать, и решение не изменится.

Проверить активность легко: подставьте решение в ограничения и посмотрите, какие разности $h - g^T \boldsymbol x_\star$ равны нулю (с точностью до погрешности солвера, например $10^{-6}$):

In [ ]:
G_np = np.array(G)
x_star = np.array(sol['x']).ravel()

for i, (g_row, h_i) in enumerate(zip(G_np, np.array(h).ravel())):
    slack = h_i - g_row @ x_star
    status = 'АКТИВНО' if abs(slack) < 1e-6 else 'неактивно'
    print(f'ограничение {i}:  h - g^T x* = {slack:8.4f}  ->  {status}')

### 1.8. Визуализация QP-задачи (пригодится в задании 1)

Нарисуем для двумерной QP-задачи линии уровня функции и серым цветом — область, в которой ограничения **выполнены**. Обратите внимание на приём: сравнение массива с числом (`xx1 + xx2 <= 5`) даёт **булев массив** той же формы, а булевы массивы комбинируются операторами `&` и `|` — так мы «раскрашиваем» допустимую область:

In [ ]:
x1 = np.linspace(-3, 5, 100)
x2 = np.linspace(-2, 6, 100)
xx1, xx2 = np.meshgrid(x1, x2)

f = -8*xx1 - 16*xx2 + xx1**2 + 4*xx2**2                      # оптимизируемая функция
ok = (xx1 + xx2 <= 5) & (xx1 >= 0) & (xx1 <= 3) & (xx2 >= 0)  # допустимая область

plt.figure(figsize=(5.5, 5.5))
plt.grid(True)
plt.xlabel('$x_1$')
plt.ylabel('$x_2$')
plt.contour(x1, x2, f, 50, alpha=0.5)
plt.contourf(x1, x2, ok, 1, alpha=0.5, cmap='gray')
plt.scatter([3], [2], color='red', zorder=3)  # точка (3, 2): проверьте, что солвер найдёт её же
plt.show()

<h2 style="color:#b45309; border-bottom:3px solid #f0a500; padding-bottom:6px; margin-top:26px;">📌 Задание 1. Линейное и квадратичное программирование</h2>

**Часть A. Линейное программирование**

1. Мебельная мастерская выпускает столы ($x_1$) и стулья ($x_2$). Прибыль: 3 и 2 тыс. руб. за изделие. Цех сборки тратит на стол 2 часа, на стул — 1 час, всего доступно 6 часов; отделка — по 1 часу на любое изделие, всего 4 часа. Составьте задачу LP, переведите максимизацию прибыли в минимизацию и решите её с помощью `solvers.lp`.
2. Нарисуйте допустимую область, линии уровня прибыли и решение (по образцу п. 1.3). Проверьте ответ, перебрав все вершины многоугольника: каждую вершину найдите как решение системы двух линейных уравнений и подставьте в прибыль.
3. Подберите другие коэффициенты прибыли, при которых оптимальным станет **целое ребро** многоугольника, а не одна вершина. Что вернёт солвер в этом случае?

**Часть B. Квадратичное программирование**

4. Решите с помощью `solvers.qp` задачу
$$f(\boldsymbol x) = -8x_1 - 16x_2 + x_1^2 + 4x_2^2 \to \min, \qquad
x_1 + x_2 \le 5, \quad 0 \le x_1 \le 3, \quad x_2 \ge 0.$$
   Сначала найдите минимум **без ограничений** (приравняв градиент к нулю) и убедитесь, что он нарушает ограничения. Затем решите задачу с ограничениями и отметьте решение на картинке из п. 1.8.
5. Какие из ограничений **активны**, а какие можно выбросить без изменения решения? Проверьте экспериментально: решите задачу, удалив «лишнее» ограничение, и сравните ответы.
6. Сравните части A и B: в какой задаче оптимум обязан лежать на границе, а в какой может оказаться внутри области? Как это связано с видом целевой функции («плоскость» против «чаши»)?

## 2. Линейный SVM

### 2.1. Разделяющая гиперплоскость и зазор

Постановка задачи: есть обучающая выборка $\{(\boldsymbol x_n, y_n)\}_{n=1}^N$, где $\boldsymbol x_n \in \mathbb{R}^d$ — вектор признаков объекта, а метка класса $y_n \in \{+1, -1\}$. Нужно по выборке построить правило, отличающее классы.

SVM ищет разделяющую гиперплоскость

$$f(\boldsymbol x) = \boldsymbol w^T \boldsymbol x + b, \qquad \hat{y}(\boldsymbol x) = \operatorname{sign} f(\boldsymbol x),$$

где $\boldsymbol w \in \mathbb{R}^d$ — вектор весов, $b$ — сдвиг. Интуиция метода: между классами есть «коридор» (**зазор**, margin), и чем он шире, тем увереннее классификатор ведёт себя на новых данных. Границы коридора — это линии уровня $f(\boldsymbol x) = +1$ и $f(\boldsymbol x) = -1$, а его ширина равна $\dfrac{2}{\lVert\boldsymbol w\rVert}$. Поэтому «сделать зазор максимальным» — то же самое, что «сделать $\lVert\boldsymbol w\rVert$ минимальным».

### 2.2. Если выборку нельзя разделить идеально: переменные $\xi_n$

В идеальном мире классы разделяются прямой, и все объекты лежат вне коридора, каждый на своей стороне. На практике объекты «залезают» в коридор или на чужую сторону. Поэтому каждому объекту разрешается **нарушать правила** — за плату. Вводится переменная ослабления (slack variable) $\xi_n \ge 0$ — величина нарушения, измеренная в долях полуширины коридора:

- $\xi_n = 0$ — объект лежит «по правилам»: строго вне коридора, на своей стороне;
- $0 < \xi_n < 1$ — объект внутри коридора, но всё ещё на своей стороне границы;
- $\xi_n > 1$ — объект ошибается: лежит на чужой стороне.

Обучение SVM — это оптимизационная задача по переменным $\boldsymbol w$, $b$, $\boldsymbol \xi$:

$$
\begin{gathered}
\frac{1}{2} \lVert\boldsymbol w\rVert^2 + C \sum_{n=1}^N \xi_n \to \min \\
\text{при: } \quad y_n (\boldsymbol w^T \boldsymbol x_n + b) \ge 1 - \xi_n, \qquad \xi_n \ge 0, \qquad n = 1, \dots, N
\end{gathered}
$$

Первое слагаемое штрафует узкий коридор (минимизируем $\lVert\boldsymbol w\rVert$ = максимизируем зазор), второе — суммарное нарушение границ. Гиперпараметр $C$ — «цена» единицы нарушения:

- **большое $C$** → нарушения «дорогие» → модель стремится не ошибаться на обучении даже ценой узкого коридора; растёт риск переобучения;
- **маленькое $C$** → коридор шире, часть объектов ошибается; при слишком малом $C$ модель недообучается.

$C$ фиксируется **до** обучения и не оптимизируется вместе с параметрами модели. Подбирают его на отложенной выборке или кросс-валидации; поскольку порядок оптимального $C$ заранее неизвестен, значения перебирают по логарифмической сетке: $10^{-3}, 10^{-2}, \dots, 10^{5}$.

### 2.3. Опорные векторы

После решения задачи $(\boldsymbol w_\star, b_\star, \boldsymbol \xi_\star)$ часть ограничений становится **активной** — обращается в точное равенство:

$$y_n (\boldsymbol w_\star^T \boldsymbol x_n + b_\star) = 1 - \xi_{\star,n}.$$

Объекты с активными ограничениями называются **опорными векторами**: это объекты, которые «держат» коридор — уберите все остальные, и решение не изменится. На практике активность проверяют с небольшим допуском, например $\lvert\, y_n f(\boldsymbol x_n) - (1 - \xi_n) \,\rvert < 10^{-5}$.

### 2.4. Сведение к задаче QP

Задача из п. 2.2 — это QP-задача, просто переменные «спрятаны» в одном длинном векторе $\boldsymbol z = (\boldsymbol w, \, b, \, \boldsymbol \xi)$ длины $d + 1 + N$. Осталось записать её в стандартной форме из п. 1.4. Сопоставьте (и проверяйте себя размерами!):

| Элемент задачи | Что получится |
|:---:|:---:|
| Квадратичная часть $\frac{1}{2}\boldsymbol z^T P \boldsymbol z = \frac{1}{2}\lVert\boldsymbol w\rVert^2$ | $P = \operatorname{diag}(\underbrace{1, \dots, 1}_{d}, \underbrace{0, \dots, 0}_{1+N})$ — единицы только для компонент $\boldsymbol w$ |
| Линейная часть $\boldsymbol q^T \boldsymbol z = C \sum_n \xi_n$ | $q = (\underbrace{0, \dots, 0}_{d+1}, \underbrace{C, \dots, C}_{N})$ |
| Правильная классификация: $y_n(\boldsymbol w^T \boldsymbol x_n + b) \ge 1 - \xi_n$, то есть $-y_n(\boldsymbol w^T \boldsymbol x_n + b) - \xi_n \le -1$ | $N$ строк матрицы $G$: $(-y_n \boldsymbol x_n^T, \; -y_n, \; -\boldsymbol e_n^T)$; соответствующие элементы $h_n = -1$ |
| Неотрицательность: $\xi_n \ge 0$, то есть $-\xi_n \le 0$ | ещё $N$ строк $G$: $(\boldsymbol 0, \; 0, \; -\boldsymbol e_n^T)$; элементы $h_n = 0$ |

Здесь $\boldsymbol e_n$ — строка из нулей с единицей на позиции $n$. Итого $G$ имеет размер $2N \times (d + 1 + N)$. Собирать такие матрицы удобно блоками: `np.hstack` / `np.vstack`, либо создать нулевую матрицу нужного размера и заполнить срезы.

**Проверка на крошечном примере ($N = 2$ объекта, $d = 2$ признака):** строка $G$ для первого объекта будет равна $(-y_1 x_{11}, \; -y_1 x_{12}, \; -y_1, \; -1, \; 0)$ — два признака, затем $b$, затем $\xi_1, \xi_2$.

### 2.5. Явное преобразование признаков

Линейная граница — грубая модель: часто классы разделимы, но не прямой. Приём: добавить к признакам их нелинейные функции, например все попарные произведения

$$\phi(x_1, x_2) = (x_1, \; x_2, \; x_1^2, \; x_1 x_2, \; x_2^2),$$

и обучить **линейный** SVM уже в новом признаковом пространстве $\boldsymbol x \mapsto \phi(\boldsymbol x)$. В исходном пространстве граница окажется кривой: уравнение $\boldsymbol w^T \phi(\boldsymbol x) + b = 0$ — квадратное уравнение относительно $x_1, x_2$, и его решением может быть, например, эллипс.

Посмотрите на анимацию: слева два «кольца» классов, которые невозможно разделить прямой; справа — те же точки после добавления всего одной нелинейной функции $z = x_1^2 + x_2^2$. В трёхмерном пространстве классы разделяются обычной плоскостью, а её «проекция» обратно на плоскость — окружность:

<img src="kernel_trick.gif" width="720">

<h2 style="color:#b45309; border-bottom:3px solid #f0a500; padding-bottom:6px; margin-top:26px;">📌 Задание 2. Линейный SVM</h2>

1. Сведите задачу обучения линейного SVM к QP (п. 2.4) и реализуйте обучение в классе `LinearSVM` (скелет ниже) при помощи QP-солвера. **Не забудьте:** солверу нужны метки $y_n \in \{+1, -1\}$; если метки закодированы нулями и единицами, преобразуйте их: `y = 2 * y - 1`.
2. Сгенерируйте три случайные двумерные выборки для бинарной классификации:
   - с линейно разделимыми классами;
   - с хорошо разделимыми классами, но **не линейно**;
   - с плохо разделимыми классами по имеющимся признакам.
3. Протестируйте линейный SVM на этих выборках. Покажите на плоскости разделяющую прямую и границы коридора $f(\boldsymbol x) = \pm 1$. Выделите опорные векторы точками другой формы или большего размера. Постройте классификаторы с разными значениями параметра $C$.
4. Как зависит число опорных векторов от параметра $C$ для каждой из выборок? Объясните, почему.
5. Используя явное преобразование признаков (п. 2.5), обучите методом опорных векторов квадратичную разделяющую поверхность и покажите её на плоскости.

### Подсказка: генерация выборок

В `scikit-learn` есть удобные генераторы:

- [`make_blobs`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_blobs.html) — два «облака» точек: линейно разделимый случай;
- [`make_moons`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_moons.html) — две «полумесяца»: классы разделимы, но не линейно;
- [`make_classification`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_classification.html) — гибкая настройка; параметр `flip_y` «переворачивает» долю меток и делает классы плохо разделимыми.

Всегда фиксируйте `random_state`, чтобы выборка не менялась при перезапуске ноутбука:

In [ ]:
from sklearn.datasets import make_blobs, make_moons, make_classification

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

X1, y1 = make_blobs(n_samples=200, centers=[(-2, 2), (2, -2)], random_state=1)
X2, y2 = make_moons(n_samples=200, noise=0.1, random_state=1)
X3, y3 = make_classification(n_samples=200, n_features=2, n_redundant=0,
                             n_clusters_per_class=1, flip_y=0.15, random_state=1)

for ax, X, y, title in zip(axes, [X1, X2, X3], [y1, y2, y3],
                           ['линейно разделимые', 'разделимы, но не линейно', 'плохо разделимые']):
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='autumn', edgecolor='black')
    ax.set_title(title)

plt.show()

### Подсказка: визуализация решающего правила

Стандартный приём: покрыть плоскость сеткой точек, применить к каждой модель и нарисовать результат фоном (`contourf`), а линию уровня $f(\boldsymbol x) = 0$ и границы коридора $f(\boldsymbol x) = \pm 1$ — линиями (`contour`). Ниже готовая функция; она ожидает от модели метод `decision_function(X)`, возвращающий числа $f(\boldsymbol x)$:

In [ ]:
def plot_decision_boundary(model, X, y, support_vectors=None, figsize=(6, 6)):
    # рисует точки выборки, разделяющую поверхность f(x) = 0 (сплошная линия)
    # и границы коридора f(x) = ±1 (пунктир)
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                         np.linspace(y_min, y_max, 300))
    zz = model.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    plt.figure(figsize=figsize)
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap='autumn', edgecolor='black', zorder=2)
    plt.contourf(xx, yy, zz, cmap='autumn', alpha=0.2, zorder=0)
    plt.contour(xx, yy, zz, levels=[-1, 0, 1], colors='k',
                linewidths=[1, 2, 1], linestyles=['--', '-', '--'], zorder=1)
    if support_vectors is not None and len(support_vectors) > 0:
        plt.scatter(support_vectors[:, 0], support_vectors[:, 1], s=220,
                    facecolors='none', edgecolors='black', linewidths=1.5, zorder=3)
    plt.tight_layout()

### Скелет класса `LinearSVM`

Чтобы ваша модель «дружила» с инструментами `scikit-learn` (`cross_val_score`, `GridSearchCV`, пайплайны и т.д.), достаточно наследоваться от [`BaseEstimator`](https://scikit-learn.org/stable/developers/develop.html) и реализовать методы `fit` и `predict`:

In [ ]:
from sklearn.base import BaseEstimator

class LinearSVM(BaseEstimator):
    def __init__(self, C=1.0, transform=None):
        self.C = C                  # цена нарушения границы
        self.transform = transform  # необязательное преобразование признаков phi(x)

    def fit(self, X, y):
        # обучение: собрать матрицы QP-задачи (п. 2.4) и вызвать solvers.qp()
        if self.transform is not None:
            X = self.transform(X)
        # TODO:
        # 1) собрать P, q, G, h для переменных z = (w, b, xi)
        # 2) sol = solvers.qp(P, q, G, h)
        # 3) разобрать решение: self.weights_ (d,), self.bias_ (число)
        # 4) найти опорные векторы и сохранить их в self.support_vectors_
        raise NotImplementedError()

    def decision_function(self, X):
        # числа f(x) = w^T x + b — нужны для рисования контуров f(x) = 0, ±1
        if self.transform is not None:
            X = self.transform(X)
        return (X @ self.weights_ + self.bias_).ravel()

    def predict(self, X):
        return np.sign(self.decision_function(X))

## 3. Двойственный переход и ядровой SVM

### 3.1. Двойственная задача

Задачу из раздела 2 называют **прямой**. У любой задачи оптимизации с ограничениями есть **двойственная** задача Лагранжа (см. [двойственность в оптимизации](https://en.wikipedia.org/wiki/Duality_(optimization)) и конспект лекций): в ней оптимизируются *двойственные переменные* — множители Лагранжа $\alpha_n$, по одному на каждый объект выборки. Для выпуклых задач QP, к которым относится SVM, оптимум двойственной задачи совпадает с оптимумом прямой, но двойственная форма удобнее: в ней признаки входят только через скалярные произведения, а это открывает путь к ядрам (п. 3.3).

Двойственная задача SVM:

$$
\begin{gathered}
\sum_{n} \alpha_n - \frac{1}{2}\sum_{n}\sum_{n'} \alpha_{n}\alpha_{n'}\, y_{n}y_{n'}\, \boldsymbol x_{n}^T\boldsymbol x_{n'} \to \max_{\boldsymbol\alpha} \\
\text{при: } \quad 0 \le \alpha_n \le C, \quad n = 1, \dots, N; \qquad \sum_{n} \alpha_n y_n = 0
\end{gathered}
$$

Как её читать:

- переменная $\alpha_n$ «прикреплена» к объекту $\boldsymbol x_n$ и играет роль «нажима» объекта на границу коридора;
- объект — **опорный вектор** тогда и только тогда, когда $\alpha_n > 0$: только эти объекты влияют на границу;
- верхняя граница $\alpha_n \le C$ — то же самое «ограничение цены ошибки» $C$, что и в прямой задаче.

Чтобы отдать задачу солверу, перепишем её как **минимизацию** $\frac{1}{2}\boldsymbol\alpha^T P \boldsymbol\alpha + \boldsymbol q^T \boldsymbol\alpha$ (проверяйте размеры!):

- $P_{nn'} = y_n y_{n'} \, \boldsymbol x_n^T \boldsymbol x_{n'}$, матрица $N \times N$;
- $q = (-1, \dots, -1)^T$;
- двойные неравенства $0 \le \alpha_n \le C$ — это $N$ пар неравенств $-\alpha_n \le 0$ и $\alpha_n \le C$: матрица $G = \begin{pmatrix} -I \\ I \end{pmatrix}$ размера $2N \times N$ и вектор $h = (0, \dots, 0, \; C, \dots, C)^T$;
- равенство $\sum_n \alpha_n y_n = 0$ — это $A = \boldsymbol y^T$ (одна строка) и $b = (0)$.

### 3.2. Предсказание и восстановление $b$

Предсказание для нового объекта $\boldsymbol x$ вычисляется через скалярные произведения с объектами обучающей выборки:

$$\hat{y}(\boldsymbol x) = \operatorname{sign}\left(\sum_{n} \alpha_n y_n \, \boldsymbol x^T \boldsymbol x_n + b\right),$$

причём суммировать достаточно только по опорным векторам — у остальных $\alpha_n = 0$.

Параметр $b$ из двойственной задачи напрямую не находится, но для любого опорного объекта, классифицируемого без ошибки, верно

$$y_n = \sum_{n'} \alpha_{n'} y_{n'} \, \boldsymbol x_n^T \boldsymbol x_{n'} + b,$$

откуда для одного такого объекта $b = y_n - \sum_{n'} \alpha_{n'} y_{n'} \boldsymbol x_n^T \boldsymbol x_{n'}$. Если есть ошибки классификации, значение $b$ **усредняют по всем опорным векторам**:

$$b = \frac{1}{|\mathrm{SV}|} \sum_{n \in \mathrm{SV}} \left(y_n - \sum_{n'} \alpha_{n'} y_{n'} \boldsymbol x_n^T \boldsymbol x_{n'}\right).$$

Интуиция: суммарные ошибки «в положительную» и «в отрицательную» стороны примерно компенсируются, и усреднение даёт устойчивую оценку. На практике: возьмите объекты с $\alpha_n > \varepsilon$ (например, $\varepsilon = 10^{-5}$) и усредните по ним.

### 3.3. Ядра, или неявное преобразование признаков

Ключевое наблюдение: и в задаче п. 3.1, и в правиле предсказания признаки входят **только в виде скалярных произведений** $\boldsymbol x^T \boldsymbol x'$. Значит, вместо явного вычисления $\phi(\boldsymbol x)$ можно всюду подставить функцию

$$k(\boldsymbol x, \boldsymbol x') = \phi(\boldsymbol x)^T \phi(\boldsymbol x'),$$

называемую **ядром**. Ядро — «мера похожести» объектов: заменив все произведения $\boldsymbol x_n^T \boldsymbol x_{n'}$ на $k(\boldsymbol x_n, \boldsymbol x_{n'})$, получаем SVM с нелинейной разделяющей поверхностью, хотя решается всё та же QP-задача. Вычислять $\phi$ явно не нужно (а её образ может лежать в пространстве огромной или даже бесконечной размерности) — в этом и состоит **ядровой трюк**.

Стандартные ядра:

- линейное: $k(\boldsymbol x, \boldsymbol x') = \boldsymbol x^T \boldsymbol x'$;
- полиномиальное степени $p$: $k(\boldsymbol x, \boldsymbol x') = (\boldsymbol x^T \boldsymbol x' + 1)^p$;
- гауссовское (RBF): $k(\boldsymbol x, \boldsymbol x') = \exp(-\gamma \lVert\boldsymbol x - \boldsymbol x'\rVert^2)$; чем меньше $\gamma$, тем «глаже» граница.

Совет: матрицу ядра $K$ с элементами $K_{nn'} = k(\boldsymbol x_n, \boldsymbol x_{n'})$ посчитайте один раз до обучения и переиспользуйте в `decision_function`. Для больших $N$ матрицы QP-задачи получаются разреженными — cvxopt умеет работать с [разреженными матрицами](https://cvxopt.org/userguide/matrices.html).

In [ ]:
def linear_kernel(X1, X2):
    # k(x, x') = x^T x';  возвращает матрицу (n1, n2)
    return X1 @ X2.T

def poly_kernel(X1, X2, p=2):
    # k(x, x') = (x^T x' + 1)^p
    return (X1 @ X2.T + 1.0) ** p

def rbf_kernel(X1, X2, gamma=1.0):
    # k(x, x') = exp(-gamma * ||x - x'||^2)
    sq1 = np.sum(X1 ** 2, axis=1)[:, None]   # ||x||^2  столбцом, (n1, 1)
    sq2 = np.sum(X2 ** 2, axis=1)[None, :]   # ||x'||^2 строкой, (1, n2)
    return np.exp(-gamma * (sq1 + sq2 - 2.0 * X1 @ X2.T))

# проверка: на диагонали RBF-матрицы стоят единицы (расстояние до себя равно нулю)
K = rbf_kernel(X1, X1, gamma=1.0)
print('размер:', K.shape, '| диагональ:', np.diag(K)[:5])

### Скелет класса `DualSVM`

Интерфейс тот же, что у `LinearSVM`: `fit` обучает, `decision_function` возвращает $\sum_n \alpha_n y_n k(\boldsymbol x, \boldsymbol x_n) + b$, `predict` — его знак:

In [ ]:
from sklearn.base import BaseEstimator

class DualSVM(BaseEstimator):
    def __init__(self, C=1.0, kernel=None):
        self.C = C
        self.kernel = kernel if kernel is not None else linear_kernel

    def fit(self, X, y):
        # обучение через двойственную QP-задачу (п. 3.1); переменные — alpha
        # TODO:
        # 1) сохранить выборку (self.X_, self.y_) и посчитать матрицу ядра K (N, N)
        # 2) собрать P, q, G, h, A, b (п. 3.1)
        # 3) sol = solvers.qp(P, q, G, h, A, b)
        # 4) сохранить alpha_; найти опорные векторы (alpha > 1e-5) -> self.support_vectors_
        # 5) вычислить self.bias_ усреднением по опорным векторам (п. 3.2)
        raise NotImplementedError()

    def decision_function(self, X):
        # f(x) = sum_n alpha_n y_n k(x, x_n) + b
        # подсказка: посчитайте матрицу ядра между X и обучающей выборкой self.X_
        raise NotImplementedError()

    def predict(self, X):
        return np.sign(self.decision_function(X))

<h2 style="color:#b45309; border-bottom:3px solid #f0a500; padding-bottom:6px; margin-top:26px;">📌 Задание 3. Двойственный SVM и ядра</h2>

1. Реализуйте процедуру обучения ядрового SVM (`DualSVM`) при помощи QP-солвера.
2. Протестируйте ядровой SVM на случайных двумерных выборках (удобно взять те же три, что и в задании 2). Покажите на плоскости разделяющие поверхности, границы коридора $f(\boldsymbol x) = \pm 1$ и опорные векторы. Попробуйте ядра:
   - линейное: $k(x, x') = x^T x'$;
   - полиномиальное: $k(x, x') = (x^T x' + 1)^p$ с различными степенями $p = 2, 3, \dots$;
   - гауссовское RBF: $k(x, x') = \exp(-\gamma \lVert x - x'\rVert^2)$.
3. Как ведёт себя SVM с полиномиальным ядром при увеличении $C$ и степени ядра $p$?
4. Как ведёт себя SVM с RBF-ядром при вариации $C$ и $\gamma$ (поварьируйте параметры по логарифмической сетке)? Какие значения параметров приводят к переобучению, а какие — к слишком грубой модели? На выборке «полумесяцы» сравните линейное, полиномиальное и RBF-ядра.


## 4. Практика: SVM на реальных данных (диагностика болезней сердца)

До сих пор все данные были синтетическими: два признака, никаких пропусков, одинаковый масштаб. Реальные данные устроены иначе: в них нет строки заголовка, пропуски закодированы символом `?`, часть признаков — категории, а масштабы различаются на порядки. В этом задании вы пройдёте полный цикл работы аналитика: **загрузка → разведка → подготовка → обучение и настройка собственной модели**.

Данные — [Heart Disease, клиника Кливленда](https://archive.ics.uci.edu/dataset/45/heart+disease) из репозитория UCI: результаты обследования 303 пациентов. По 13 признакам (возраст, давление, холестерин, результаты ЭКГ и стресс-тестов и др.) нужно предсказать, есть ли у пациента заболевание сердца, — бинарная классификация с понятным прикладным смыслом: такие модели используются как скрининговая поддержка решений врача.

### 4.1. Загрузка и разведка данных (`pandas`)

Оригинальный файл распространяется **без строки заголовка**, а пропуски в нём обозначены символом `?` — с этим и разберётся `pandas`. Словарь признаков:

| Признак | Описание |
|:---:|:---|
| `age` | возраст, лет |
| `sex` | пол (1 — мужской, 0 — женский) |
| `cp` | тип грудной боли (1–4) |
| `trestbps` | артериальное давление в покое, мм рт. ст. |
| `chol` | уровень холестерина, мг/дл |
| `fbs` | сахар в крови натощак выше 120 мг/дл (1 — да, 0 — нет) |
| `restecg` | результаты ЭКГ в покое (0–2) |
| `thalach` | достигнутая максимальная частота пульса |
| `exang` | стенокардия, вызванная нагрузкой (1 — да, 0 — нет) |
| `oldpeak` | депрессия сегмента ST на ЭКГ при нагрузке |
| `slope` | наклон пикового сегмента ST (1–3) |
| `ca` | число крупных сосудов, окрашенных при флуороскопии (0–3) |
| `thal` | результат сцинтиграфии с таллием: 3 — норма, 6 — фиксированный дефект, 7 — обратимый дефект |
| `num` | **целевой признак**: диагноз (0 — здоров, 1–4 — степень заболевания) |

Загрузите таблицу функцией [`pd.read_csv()`](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) с параметрами `names=...` (имена столбцов) и `na_values='?'` (так обозначены пропуски). Минимальный набор команд разведки:

- `df.head()`, `df.shape` — первые строки и размер таблицы;
- `df.info()`, `df.isna().sum()` — типы столбцов и где пропуски;
- `df.describe()` — базовые статистики числовых столбцов;
- `df['num'].value_counts()` — сколько здоровых и больных (баланс классов).

In [ ]:
import pandas as pd

columns = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg',
           'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'num']
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data'
try:
    df = pd.read_csv(url, names=columns, na_values='?')          # загрузка из интернета
except Exception:
    df = pd.read_csv('heart.csv', names=columns, na_values='?')  # копия рядом с ноутбуком

df.head()

In [ ]:
# Начало разведки: размер данных и пропуски по столбцам
print('размер таблицы:', df.shape)
df.isna().sum()

### 4.2. Подготовка данных: чек-лист

Прежде чем подавать данные в SVM, их нужно привести к виду «числовая матрица $X$ + вектор меток $y \in \{+1, -1\}$». Пройдитесь по чек-листу (каждый шаг прокомментируйте в отчёте — что и почему вы делаете):

1. **Целевая переменная**: в `num` значения 0–4. Приведите её к бинарной метке: `y = (df['num'] > 0).astype(int)`, затем `y = 2 * y - 1` (получатся $-1$ и $+1$). Сам столбец `num` уберите из признаков.
2. **Пропуски**: их всего шесть строк в `ca` и `thal` — удалите эти строки (`dropna`) или заполните (медиана, мода). Обоснуйте выбор: сколько процентов данных вы теряете?
3. **Категориальные признаки**: `cp`, `restecg`, `slope`, `thal` — это **категории, а не числа** (большее значение не означает «больше»): закодируйте их через `pd.get_dummies(...)`. Признаки `sex`, `fbs`, `exang` уже бинарные — оставьте как есть.
4. **Разбить данные**: [`train_test_split`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) с параметрами `test_size=0.2, stratify=y, random_state=...` — тестовая выборка нужна для честной оценки качества.
5. **Масштабировать признаки**: [`StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) приводит каждый признак к нулевому среднему и единичному отклонению. Обучать скейлер нужно **только на train**, а к test применять уже обученный (`transform`). Почему масштабирование критично для SVM? Вспомните: модель строится на скалярных произведениях и норме $\lVert\boldsymbol w\rVert$, а признаки «холестерин» (сотни мг/дл) и «пол» (0/1) различаются на 2–3 порядка — без выравнивания масштаба «громкий» признак заглушит остальные.

<h2 style="color:#b45309; border-bottom:3px solid #f0a500; padding-bottom:6px; margin-top:26px;">📌 Задание 4. SVM на реальных данных</h2>

1. Выполните загрузку, разведку и подготовку данных по чек-листу п. 4.2. В отчёте покажите таблицу до и после подготовки и поясните каждый шаг.
2. Обучите **свой** `LinearSVM` (из задания 2) на подготовленных данных. Подберите $C$ по логарифмической сетке $10^{-3}, \dots, 10^{5}$: посчитайте accuracy на train и test для каждого значения и изобразите оба графика в зависимости от $\log_{10} C$. Выберите лучшее $C$ и объясните выбор.
3. Обучите **свой** `DualSVM` с RBF-ядром: подберите пару $(C, \gamma)$ по логарифмической сетке (удобно построить тепловую карту качества). Стало ли лучше, чем с линейным SVM? На какой модели и параметрах остановитесь?
4. Проверьте себя: обучите [`sklearn.svm.SVC`](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html) с `kernel='linear'` и `kernel='rbf'` на тех же данных и гиперпараметрах — качество должно получиться близким к вашему. Если различие велико — ищите ошибку в реализации.
5. Ответьте на вопросы в отчёте:
   - что произойдёт с качеством вашего SVM, если **не** масштабировать признаки? Проверьте экспериментально и объясните результат;
   - какие признаки сильнее всего влияют на диагноз? Для линейной SVM естественная мера важности признака $j$ — модуль веса $\lvert w_j \rvert$: отсортируйте признаки по важности и дайте клиническую интерпретацию — согласуется ли результат с медицинской интуицией (что важнее: боль в груди, возраст или холестерин?);
   - какую долю обучающей выборки составляют опорные векторы и как эта доля зависит от $C$? Много ли пациентов оказываются «на границе» диагноза?

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.svm import SVC

# Каркас для задания 4 (замените заглушки на свои X и y после подготовки данных):
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
#
# scaler = StandardScaler().fit(X_train)          # обучаем ТОЛЬКО на train
# X_train_scaled = scaler.transform(X_train)
# X_test_scaled = scaler.transform(X_test)
#
# model = LinearSVM(C=1.0)                        # ваша реализация
# model.fit(X_train_scaled, y_train)
# print('train accuracy:', accuracy_score(y_train, model.predict(X_train_scaled)))
# print('test  accuracy:', accuracy_score(y_test,  model.predict(X_test_scaled)))

## Итог: что должно быть в отчёте

- **Задание 1** — задачи LP и QP: план производства с картинкой и перебором вершин, тестовая QP-задача, анализ активных ограничений, сравнение LP и QP.
- **Задание 2** — три выборки, эксперименты линейного SVM с разными $C$, квадратичная граница через явное преобразование признаков.
- **Задание 3** — эксперименты ядрового SVM (линейное, полиномиальное, RBF-ядро) с выводами о влиянии $C$, $p$, $\gamma$.
- **Задание 4** — разведка и подготовка данных о болезнях сердца с пояснениями, подбор гиперпараметров своего SVM, сравнение с `sklearn.svm.SVC`, ответы на вопросы.

Код сам по себе не оценивается — оцениваются полнота ответов, корректность экспериментов и аргументированность выводов.